# Prompting Strategy Analysis
Loads results from `scripts/run_eval.py` and produces the main figures.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

from mpe.analysis import load_results, accuracy_table, mcnemar_pairwise, difficulty_correlation

RESULTS_PATH = Path("../results/latest.jsonl")
df = load_results(RESULTS_PATH)
print(f"{len(df)} rows, strategies: {df['strategy'].unique().tolist()}")
df.head()

## Accuracy Table

In [ ]:
acc = accuracy_table(df)
print(acc.to_string(float_format="{:.1%}".format))

## Figure 1 — Overall Accuracy by Strategy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

strategies = acc.index.tolist()
overall = acc["overall"].values
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

bars = ax.bar(strategies, overall, color=colors[:len(strategies)], width=0.55, edgecolor="white")
ax.bar_label(bars, fmt="{:.1%}", padding=4, fontsize=10)
ax.set_ylim(0, min(1.0, overall.max() * 1.25))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylabel("Accuracy")
ax.set_title("Overall Accuracy by Prompting Strategy")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("../results/fig1_overall_accuracy.png", dpi=150)
plt.show()

## Figure 2 — Accuracy by Difficulty Level

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

level_cols = [c for c in acc.columns if c.startswith("level_")]
levels = [int(c.split("_")[1]) for c in level_cols]

for i, strat in enumerate(strategies):
    vals = acc.loc[strat, level_cols].values.astype(float)
    ax.plot(levels, vals, marker="o", label=strat, color=colors[i])

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xlabel("Difficulty Level")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by Difficulty Level and Strategy")
ax.set_xticks(levels)
ax.legend(loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("../results/fig2_accuracy_by_level.png", dpi=150)
plt.show()

## Statistical Significance — McNemar's Test

Paired test on the same problem set. Tests whether the differences between strategies are statistically significant.

In [ ]:
baseline = "zero_shot"
comparisons = [s for s in strategies if s != baseline]

rows = []
for strat in comparisons:
    rows.append(mcnemar_pairwise(df, baseline, strat))

sig_df = pd.DataFrame(rows).set_index("strategy_b")
sig_df[["a_only", "b_only", "statistic", "pvalue", "significant_at_05"]]

## Difficulty Correlation

Pearson correlation between problem level and accuracy per strategy. A value near -1 means accuracy drops sharply with difficulty.

In [ ]:
difficulty_correlation(df)